# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

month_03 = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

# --- Same page-month aggregation and position fix as w04 ---
page_month = month_03.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
    sessions_organic=("sessions_organic", "sum"),
)

valid_position_rows = month_03[month_03["gsc_avg_position"] > 0].copy()
valid_position_rows["_weight"] = valid_position_rows["gsc_impressions"].clip(lower=1)
valid_position_rows["_weighted_pos"] = valid_position_rows["gsc_avg_position"] * valid_position_rows["_weight"]
grouped_sums = valid_position_rows.groupby(
    ["client_hash_id", "content_hash_id"], as_index=False
)[["_weighted_pos", "_weight"]].sum()
grouped_sums["avg_position"] = grouped_sums["_weighted_pos"] / grouped_sums["_weight"]
weighted_pos = grouped_sums[["client_hash_id", "content_hash_id", "avg_position"]]

page_month = page_month.merge(weighted_pos, on=["client_hash_id", "content_hash_id"], how="left")
page_month = page_month[page_month["avg_position"].notna()].copy()

page_month["ctr_pct"] = np.where(
    page_month["total_impressions"] > 0,
    page_month["total_clicks"] / page_month["total_impressions"] * 100,
    np.nan
)

# --- New: spike_day_share for EVERY page, not just the top 10 ---
month_03["day_median"] = month_03.groupby(
    ["client_hash_id", "content_hash_id"]
)["gsc_impressions"].transform("median").clip(lower=1)
month_03["is_spike_day"] = month_03["gsc_impressions"] > 5 * month_03["day_median"]

spike_share_all = month_03.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: g.loc[g["is_spike_day"], "gsc_impressions"].sum() / g["gsc_impressions"].sum()
).reset_index(name="spike_day_share")

page_month = page_month.merge(spike_share_all, on=["client_hash_id", "content_hash_id"], how="left")
page_month["spike_day_share"] = page_month["spike_day_share"].fillna(0.0)

# --- Baseline score + ctr_gap, same construction as w04 ---
MIN_IMPRESSIONS = 500
page_month["position_tier"] = pd.cut(
    page_month["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
visible = (page_month["total_impressions"] >= MIN_IMPRESSIONS).astype(int)
visible_rows = page_month[visible == 1]
tier_baseline = visible_rows.groupby("position_tier", observed=True)["ctr_pct"].median()
expected_ctr_by_tier = page_month["position_tier"].astype(str).map(tier_baseline.to_dict()).astype(float)
page_month["ctr_gap"] = expected_ctr_by_tier - page_month["ctr_pct"]

page_month["visible"] = visible
page_month["baseline_score"] = visible * page_month["total_impressions"] * page_month["ctr_gap"].clip(lower=0)

print("page_month shape:", page_month.shape)
print("Unique clients:", page_month["client_hash_id"].nunique())
page_month.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
FEATURES = ["avg_position", "total_impressions", "spike_day_share",
            "ga4_engaged_sessions", "sessions_organic"]
TARGET = "ctr_gap"

model_data = page_month.dropna(subset=FEATURES + [TARGET]).copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(model_data, groups=model_data["client_hash_id"]))

train_df = model_data.iloc[train_idx]
test_df = model_data.iloc[test_idx]

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print("Train rows:", len(train_df), "| Train clients:", len(train_clients))
print("Test rows:", len(test_df), "| Test clients:", len(test_clients))
print("Client overlap between train and test (should be 0):", len(train_clients & test_clients))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
def precision_at_k(scores, true_top_k_ids, ids, k):
    order = np.argsort(-np.asarray(scores))
    top_k_ids = np.asarray(ids)[order[:k]]
    return np.isin(top_k_ids, true_top_k_ids).mean()

K = 30

# --- Pull April and build its own independent page-month gap, same pipeline as March ---
month_04 = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

page_month_04 = month_04.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
)

valid_pos_04 = month_04[month_04["gsc_avg_position"] > 0].copy()
valid_pos_04["_weight"] = valid_pos_04["gsc_impressions"].clip(lower=1)
valid_pos_04["_weighted_pos"] = valid_pos_04["gsc_avg_position"] * valid_pos_04["_weight"]
grouped_04 = valid_pos_04.groupby(
    ["client_hash_id", "content_hash_id"], as_index=False
)[["_weighted_pos", "_weight"]].sum()
grouped_04["avg_position"] = grouped_04["_weighted_pos"] / grouped_04["_weight"]

page_month_04 = page_month_04.merge(
    grouped_04[["client_hash_id", "content_hash_id", "avg_position"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)
page_month_04 = page_month_04[page_month_04["avg_position"].notna()].copy()

page_month_04["ctr_pct"] = np.where(
    page_month_04["total_impressions"] > 0,
    page_month_04["total_clicks"] / page_month_04["total_impressions"] * 100,
    np.nan
)
page_month_04["position_tier"] = pd.cut(
    page_month_04["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
april_visible = (page_month_04["total_impressions"] >= MIN_IMPRESSIONS)
april_tier_baseline = page_month_04[april_visible].groupby(
    "position_tier", observed=True
)["ctr_pct"].median()
april_expected_ctr = page_month_04["position_tier"].astype(str).map(april_tier_baseline.to_dict()).astype(float)
page_month_04["april_ctr_gap"] = april_expected_ctr - page_month_04["ctr_pct"]
page_month_04["april_visible"] = april_visible.astype(int)

# --- Match March test-set pages to their real April outcome ---
april_lookup = page_month_04[["client_hash_id", "content_hash_id", "april_ctr_gap", "april_visible"]]
test_matched = test_visible.merge(april_lookup, on=["client_hash_id", "content_hash_id"], how="inner")
test_matched = test_matched[test_matched["april_visible"] == 1].copy()

print(f"March test-visible rows: {len(test_visible)} | matched with an April outcome: {len(test_matched)}")

test_ids = (test_matched["client_hash_id"] + "|" + test_matched["content_hash_id"]).values
true_top_k_ids = test_ids[np.argsort(-test_matched["april_ctr_gap"].values)[:K]]
base_rate = K / len(test_matched)

baseline_precision = precision_at_k(test_matched["baseline_score"].values, true_top_k_ids, test_ids, K)

tree_preds_matched = tree_model.predict(test_matched[FEATURES])
forest_preds_matched = forest_model.predict(test_matched[FEATURES])

tree_precision = precision_at_k(tree_preds_matched, true_top_k_ids, test_ids, K)
forest_precision = precision_at_k(forest_preds_matched, true_top_k_ids, test_ids, K)

comparison = pd.DataFrame({
    "method": ["base_rate (random)", "baseline (w04 rule)", "decision_tree", "random_forest"],
    f"precision@{K} (April outcome)": [base_rate, baseline_precision, tree_precision, forest_precision],
})
print(comparison.to_string(index=False))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
perm = permutation_importance(
    forest_model, X_test_visible, test_visible[TARGET],
    n_repeats=20, random_state=RANDOM_SEED
)
importance_table = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print(importance_table.to_string(index=False))

# Sanity check: does the top feature make sense, or does it look suspiciously perfect?
# avg_position ranking near the top is expected (it directly built the baseline's own tier
# lookup); an importance value well above what a stable model should show (e.g. exceeding 1,
# which is not achievable through a normal R^2 drop) signals an unstable, likely-outlier-driven
# feature rather than a genuinely dominant one -- worth checking that feature's distribution
# for extreme values before trusting it.

# --- Three largest errors, measured against the REAL April outcome, not the March
# training target -- this is the error analysis that actually matters for the forward test. ---
test_matched = test_matched.copy()
test_matched["predicted_gap"] = forest_preds_matched
test_matched["abs_error_vs_april"] = (test_matched["predicted_gap"] - test_matched["april_ctr_gap"]).abs()

worst = test_matched.sort_values("abs_error_vs_april", ascending=False).head(3)
print("\nThree largest errors (predicted March priority vs actual April gap):")
print(worst[["client_hash_id", "content_hash_id", "avg_position", "total_impressions",
             "spike_day_share", "ctr_gap", "april_ctr_gap", "predicted_gap",
             "abs_error_vs_april"]].to_string(index=False))

# For each of the 3 rows above: compare ctr_gap (March, what the model learned from) against
# april_ctr_gap (what actually happened) before concluding why the model missed -- a large
# swing between the two often means the page's situation genuinely changed month to month,
# not that the model reasoned badly from what it had in March.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.